<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-01-setup-and-iam/lesson-1.1-setup/practice/GCP_Capstone_1.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 1.1 — Setting Up GCP AI Project

8 hands-on exercises with complete solutions. Build your GCP GenAI environment from scratch and verify every component works.

Runnable companion to the published practice lab. Each exercise below shows the objective and a complete solution. Cloud Shell / `gcloud` steps are `%%bash` cells; Python steps run in Colab after you authenticate and set your project.

---

## Exercise 1: Create & Configure Your GCP Project  
**Difficulty:** Easy

Create a new GCP project, set it as your default, and verify the configuration.

1. Open console.cloud.google.com and click Activate Cloud Shell (terminal icon top-right)
2. Create a project with a unique ID (replace YOUR-NAME with your actual name)
3. Set the project as your default
4. Verify the configuration shows your project

**Solution:**

In [ ]:
%%bash
# 1. Create the project (replace YOUR-NAME)
gcloud projects create documind-ai-YOUR-ID   --name="DocuMind AI Capstone"

# 2. Set as default project
gcloud config set project documind-ai-YOUR-ID

# 3. Verify configuration
gcloud config list

# 4. Check project details
gcloud projects describe documind-ai-YOUR-ID

## Exercise 2: Link Billing & Set Budget Alerts  
**Difficulty:** Easy

Connect your billing account (free trial or startup credits) and create a 3-tier budget alert to prevent surprise charges.

1. Find your billing account ID
2. Link it to your project
3. Create budget alerts at 50% ($12.50), 80% ($20), and 100% ($25)
4. Verify billing is active

**Solution:**

In [ ]:
%%bash
# 1. Find your billing account
gcloud billing accounts list
# Note the ACCOUNT_ID (format: XXXXXX-XXXXXX-XXXXXX)

# 2. Link billing to project
gcloud billing projects link documind-ai-YOUR-ID   --billing-account=XXXXXX-XXXXXX-XXXXXX

# 3. Verify billing is linked
gcloud billing projects describe documind-ai-YOUR-ID
# Should show: billingEnabled: true

# 4. Create budget alert ($25 budget, alerts at 50%, 80%, 100%)
# Via Console: Billing > Budgets & alerts > Create Budget
# Set budget: $25
# Thresholds: 50% ($12.50), 80% ($20), 100% ($25)

# Or via CLI (requires billingbudgets API):
gcloud services enable billingbudgets.googleapis.com
gcloud billing budgets create   --billing-account=XXXXXX-XXXXXX-XXXXXX   --display-name="DocuMind Safety Net"   --budget-amount=25   --threshold-rule=percent=0.5   --threshold-rule=percent=0.8   --threshold-rule=percent=1.0

## Exercise 3: Enable All GenAI APIs  
**Difficulty:** Easy

Enable 15 APIs needed for the capstone project in a single batch command, then verify all are active.

1. Run the batch enable command
2. Wait 60 seconds for propagation
3. List all enabled APIs and count them

**Solution:**

In [ ]:
%%bash
# 1. Batch enable all APIs
gcloud services enable   aiplatform.googleapis.com   run.googleapis.com   firestore.googleapis.com   cloudbuild.googleapis.com   secretmanager.googleapis.com   storage.googleapis.com   documentai.googleapis.com   speech.googleapis.com   vision.googleapis.com   language.googleapis.com   translate.googleapis.com   bigquery.googleapis.com   artifactregistry.googleapis.com   monitoring.googleapis.com   logging.googleapis.com

# 2. Wait for propagation
echo "Waiting 60s for API propagation..."
sleep 60

# 3. List and count enabled APIs
gcloud services list --enabled   --format="table(name,title)"

# 4. Count total
echo "Total APIs enabled:"
gcloud services list --enabled   --format="value(name)" | wc -l

## Exercise 4: Project Structure & Virtual Environment  
**Difficulty:** Medium

Create the complete DocuMind project folder structure, initialize Git, create a .gitignore that blocks credential files, and set up a Python virtual environment.

1. Create the folder structure
2. Initialize a Git repository
3. Create a comprehensive .gitignore
4. Set up Python venv and install dependencies
5. Verify everything is working

**Solution:**

In [ ]:
%%bash
# 1. Create project structure
mkdir -p ~/documind-ai/{src/services,src/routers,tests,deploy,notebooks}
cd ~/documind-ai

# 2. Initialize Git
git init
echo "# DocuMind AI - GCP GenAI Capstone" > README.md

# 3. Create .gitignore
cat > .gitignore << 'EOF'
# Virtual environments
venv/
.venv/

# Python
__pycache__/
*.py[cod]
*.egg-info/

# CRITICAL: Never commit credentials
.env
.env.*
*-credentials*.json
*-sa-key*.json
*.tfstate
.terraform/

# IDE and OS
.vscode/
.idea/
.DS_Store

# Model artifacts
*.pkl
*.h5
*.pt

# Jupyter
.ipynb_checkpoints/
EOF

# 4. Create requirements.txt
cat > requirements.txt << 'EOF'
google-genai>=2.0,<3.0
google-cloud-aiplatform>=1.70
google-cloud-firestore>=2.19
google-cloud-storage>=2.19
google-cloud-bigquery>=3.26
google-cloud-secret-manager>=2.21
google-cloud-logging>=3.11
fastapi[standard]>=0.115
python-dotenv>=1.0
pydantic>=2.10
EOF

# 5. Create Python virtual environment
python3 -m venv venv
source venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt

# 6. Create __init__.py files
touch src/__init__.py src/services/__init__.py src/routers/__init__.py tests/__init__.py

# 7. Verify structure
tree -I venv
# If tree not installed: find . -not -path './venv/*' | head -20

# 8. First Git commit
git add .
git commit -m "feat: initial project structure with GCP dependencies"

## Exercise 5: First Gemini API Call & Cost Analysis  
**Difficulty:** Medium

Make your first Gemini API call, read the token usage, calculate the cost in INR, and figure out how many calls $500 in credits can buy.

1. Write verify_setup.py with the Gemini call
2. Run it and read the token counts
3. Calculate cost per call and total calls possible with $500

**Solution:**

In [ ]:
from google import genai

# Initialize Vertex AI client
client = genai.Client(
    enterprise=True,
    project="documind-ai-YOUR-ID",  # YOUR project ID
    location="global"
)

# Make the call
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain what Google Cloud Platform is in 2 sentences."
)

# Print response
print("✅ Gemini says:")
print(response.text)

# Token analysis
u = response.usage_metadata
input_cost = u.prompt_token_count * 1.50 / 1_000_000
output_cost = u.candidates_token_count * 7.50 / 1_000_000
total_cost = input_cost + output_cost

print(f"\n📊 Tokens: {u.prompt_token_count} in + {u.candidates_token_count} out = {u.total_token_count} total")
print(f"💰 Cost: ${total_cost:.6f} (₹{total_cost*85:.4f})")

# How many calls with $500?
calls_possible = 500 / total_cost
print(f"\n💪 $500 budget = {calls_possible:,.0f} calls at this rate")
print(f"   That's {calls_possible/30:,.0f} calls per day for 30 days")

## Exercise 6: Compare Three Gemini Models  
**Difficulty:** Medium

Call Flash-Lite, Flash, and Pro with the same prompt. Compare response quality, token usage, latency, and cost. Understand when to use each model.

**Solution:**

In [ ]:
import time
from google import genai

client = genai.Client(enterprise=True, project="documind-ai-YOUR-ID", location="global")

prompt = "Explain the difference between supervised and unsupervised learning. Give one example of each."

# Model configs: (name, input_price_per_M, output_price_per_M)
models = [
    ("gemini-3.1-flash-lite", 0.25, 1.50),
    ("gemini-3.6-flash", 1.50, 7.50),
    ("gemini-3.1-pro-preview", 2.00, 12.00),
]

print("📊 Model Comparison")
print("=" * 70)

for model_name, in_price, out_price in models:
    start = time.time()
    try:
        resp = client.models.generate_content(model=model_name, contents=prompt)
        latency = (time.time() - start) * 1000
        u = resp.usage_metadata
        cost = (u.prompt_token_count * in_price + u.candidates_token_count * out_price) / 1_000_000

        print(f"\n📌 {model_name}")
        print(f"   Tokens:  {u.prompt_token_count} in + {u.candidates_token_count} out = {u.total_token_count}")
        print(f"   Latency: {latency:.0f}ms")
        print(f"   Cost:    ${cost:.6f} (₹{cost*85:.4f})")
        print(f"   Preview: {resp.text[:120]}...")
    except Exception as e:
        print(f"\n❌ {model_name}: {e}")

## Exercise 7: Global-Endpoint Latency  
**Difficulty:** Challenge

Gemini 3.x generation is served **only** from the `global` endpoint — Google routes each request to the nearest available capacity. A regional client (`location="us-central1"` or `"asia-south1"`) now returns **404 “model not found in the specified region”**, so the old “compare us-central1 vs asia-south1 generation latency” test no longer applies. Instead, measure the latency you actually get from the global endpoint over 10 calls.

Region choice still matters — but for **data-residency services** (embeddings, Cloud Run, Storage, BigQuery), which stay regional. For Indian data you keep those in `asia-south1`.

In [ ]:
import time
from google import genai

# Gemini 3.x generation is GLOBAL-ONLY: a regional client (us-central1 or
# asia-south1) returns 404 "model not found in the specified region". So we no
# longer compare per-region generation latency — we measure the global endpoint, which
# Google routes to the nearest available capacity. Regional endpoints still matter for
# data-residency services (embeddings, Cloud Run, Storage, BigQuery) — keep those in
# asia-south1 for Indian data.
prompt = "What is 2+2?"  # Simple prompt for consistent timing
NUM_CALLS = 10

client = genai.Client(enterprise=True, project="documind-ai-YOUR-ID", location="global")
latencies = []

print("🌐 Measuring global-endpoint latency...")
for i in range(NUM_CALLS):
    start = time.time()
    try:
        resp = client.models.generate_content(model="gemini-3.6-flash", contents=prompt)
        ms = (time.time() - start) * 1000
        latencies.append(ms)
        print(f"   Call {i+1}: {ms:.0f}ms")
    except Exception as e:
        print(f"   Call {i+1}: ERROR - {e}")

if latencies:
    avg = sum(latencies) / len(latencies)
    p50 = sorted(latencies)[len(latencies)//2]
    mx = sorted(latencies)[-1]
    print(f"   📊 Avg: {avg:.0f}ms | P50: {p50:.0f}ms | Max: {mx:.0f}ms")
    print("   ℹ️  Region no longer affects Gemini 3.x generation (global-only);")
    print("      it still applies to data-residency services like embeddings & Cloud Run.")
else:
    print("   ❌ No successful calls")

## Exercise 8: Monthly Cost Calculator  
**Difficulty:** Challenge

Build a Python function that estimates monthly Gemini API cost given daily query volume, average token counts, and model name. Test with realistic DocuMind usage patterns and answer: how long will $500 last?

**Solution:**

In [ ]:
"""Monthly Cost Calculator for Gemini API"""

# Pricing per million tokens (August 2026)
PRICING = {
    "gemini-3.1-flash-lite": {"input": 0.25, "output": 1.50},
    "gemini-3.6-flash":      {"input": 1.50,  "output": 7.50},
    "gemini-3.1-pro-preview":        {"input": 2.00,  "output": 12.00},
}

def estimate_monthly_cost(
    queries_per_day: int,
    avg_input_tokens: int,
    avg_output_tokens: int,
    model: str = "gemini-3.6-flash"
) -> dict:
    prices = PRICING[model]
    monthly = queries_per_day * 30
    input_cost = monthly * avg_input_tokens * prices["input"] / 1_000_000
    output_cost = monthly * avg_output_tokens * prices["output"] / 1_000_000
    total = input_cost + output_cost
    return {
        "model": model,
        "monthly_queries": monthly,
        "input_cost": round(input_cost, 2),
        "output_cost": round(output_cost, 2),
        "total_usd": round(total, 2),
        "total_inr": round(total * 85, 2),
        "per_query_inr": round(total * 85 / monthly, 4),
        "months_from_500": round(500 / total, 1) if total > 0 else float("inf"),
    }

# Test with DocuMind realistic usage
print("💰 DocuMind Monthly Cost Estimates")
print("=" * 60)
print(f"{'Model':<25} {'Monthly':>10} {'Per Query':>12} {'$500 lasts':>12}")
print("-" * 60)

for model in PRICING:
    r = estimate_monthly_cost(
        queries_per_day=100,    # 100 RAG queries/day
        avg_input_tokens=2000,  # System prompt + RAG context
        avg_output_tokens=500,  # Response with citations
        model=model
    )
    print(f"{r['model']:<25} ₹{r['total_inr']:>8} {r['per_query_inr']:>10}/q  {r['months_from_500']:>8} months")